# 5. Correlation and Covariance

**Statistical Foundations for Data Science — Notebook 5 of 8**

So far each notebook has looked at **one** variable at a time. But the interesting
questions are always about relationships: does study time affect marks? Does ad spend move
revenue? Which features actually carry information about the target?

**Covariance** and **correlation** are the two basic tools for measuring how two numeric
variables move together — and this notebook spends as much time on their *limitations* as
on their computation, because misread correlations are the most common statistical error in
industry.

### What you will learn

1. **Covariance**: definition, computation, and why its units make it hard to read
2. **Pearson correlation** $r$: the scale-free version, and how to interpret it
3. What $r$ **misses**: non-linearity, outliers, and Anscombe's quartet
4. **Spearman** and **Kendall** rank correlations, and when to prefer them
5. Correlation **matrices** and heatmaps
6. **Testing** whether a correlation is statistically significant
7. **Correlation ≠ causation**: confounders, reverse causation, and Simpson's paradox
8. **Partial correlation** — controlling for a third variable
9. **Multicollinearity** and the VIF, and why it wrecks regression coefficients

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

rng = np.random.default_rng(seed=5)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 5.1 Covariance

**Covariance** measures whether two variables tend to be above their means at the same
time:

$$\operatorname{Cov}(X, Y) = E\big[(X - \mu_X)(Y - \mu_Y)\big] = E[XY] - E[X]E[Y]$$

Sample version (note the $n-1$, same Bessel correction as the variance):

$$s_{xy} = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})$$

**Reading the sign:**
- $\operatorname{Cov} > 0$ — they tend to move in the same direction
- $\operatorname{Cov} < 0$ — opposite directions
- $\operatorname{Cov} \approx 0$ — no *linear* relationship

**Reading the magnitude:** you cannot. The units are (units of $X$) × (units of $Y$), so a
covariance of 5,000 between height-in-cm and weight-in-kg becomes 50,000 if you switch to
millimetres. That scale-dependence is exactly the problem correlation solves.

Note also that $\operatorname{Cov}(X, X) = \operatorname{Var}(X)$.

In [ ]:
# Build data with a known relationship
n = 200
study_hours = rng.uniform(0, 20, n)
marks = 35 + 2.5 * study_hours + rng.normal(0, 6, n)

df = pd.DataFrame({"study_hours": study_hours, "marks": marks})

# Covariance by hand, then with pandas
dx = df.study_hours - df.study_hours.mean()
dy = df.marks - df.marks.mean()
cov_manual = (dx * dy).sum() / (n - 1)

print(f"Covariance (manual) : {cov_manual:.3f}")
print(f"Covariance (pandas) : {df.study_hours.cov(df.marks):.3f}")
print(f"Covariance (numpy)  : {np.cov(df.study_hours, df.marks, ddof=1)[0, 1]:.3f}")
print("\nFull covariance matrix (diagonal = variances):")
print(df.cov().round(2))

In [ ]:
# Why the magnitude is meaningless: change the units and the number changes
variants = pd.DataFrame({
    "hours":   df.study_hours,
    "minutes": df.study_hours * 60,
    "marks":   df.marks,
    "marks_pct_of_200": df.marks / 2,
})

print("Same underlying relationship, four different covariances:")
print(f"  Cov(hours,   marks)            = {variants.hours.cov(variants.marks):10.3f}")
print(f"  Cov(minutes, marks)            = {variants.minutes.cov(variants.marks):10.3f}")
print(f"  Cov(hours,   marks_pct_of_200) = {variants.hours.cov(variants.marks_pct_of_200):10.3f}")
print(f"  Cov(minutes, marks_pct_of_200) = {variants.minutes.cov(variants.marks_pct_of_200):10.3f}")
print("\nBut correlation is identical in every case:")
for a, b in [("hours", "marks"), ("minutes", "marks"),
             ("hours", "marks_pct_of_200"), ("minutes", "marks_pct_of_200")]:
    print(f"  r({a}, {b}) = {variants[a].corr(variants[b]):.6f}")

In [ ]:
# Visualising covariance: the four quadrants around the means
fig, ax = plt.subplots(figsize=(6.5, 5))
mx, my = df.study_hours.mean(), df.marks.mean()
colours = np.where((df.study_hours - mx) * (df.marks - my) > 0, "steelblue", "crimson")
ax.scatter(df.study_hours, df.marks, c=colours, s=22, alpha=0.75)
ax.axvline(mx, color="black", lw=1); ax.axhline(my, color="black", lw=1)
ax.text(mx + 0.4, my + 14, "both above\n(+)", fontsize=9)
ax.text(mx - 6.5, my - 20, "both below\n(+)", fontsize=9)
ax.text(mx + 0.4, my - 20, "opposite\n(-)", fontsize=9, color="crimson")
ax.text(mx - 6.5, my + 14, "opposite\n(-)", fontsize=9, color="crimson")
ax.set_xlabel("study hours"); ax.set_ylabel("marks")
ax.set_title("Covariance sums the products of deviations")
plt.show()

print("Blue points push the covariance up, red points pull it down.")
print("Positive covariance simply means blue dominates.")

---
## 5.2 Pearson correlation coefficient

Divide the covariance by both standard deviations and the units cancel:

$$r = \frac{\operatorname{Cov}(X, Y)}{\sigma_X \sigma_Y}
    = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2}\sqrt{\sum (y_i - \bar{y})^2}}$$

Now $-1 \le r \le 1$ always. Equivalently, $r$ is the covariance of the **z-scores**.

### Interpreting $|r|$ (rough conventions — the field matters)

| $\|r\|$ | Strength |
|---|---|
| 0.00 – 0.19 | very weak / negligible |
| 0.20 – 0.39 | weak |
| 0.40 – 0.59 | moderate |
| 0.60 – 0.79 | strong |
| 0.80 – 1.00 | very strong |

In physics $r = 0.95$ is unremarkable; in social science $r = 0.3$ can be a major finding.

### $r^2$ — the useful reframing

$r^2$ is the **proportion of variance in $Y$ explained by a linear fit on $X$**. So
$r = 0.7$ means $r^2 = 0.49$: about half the variation is accounted for. Reporting $r^2$
keeps people honest — $r = 0.3$ sounds respectable, but it explains only 9%.

In [ ]:
# Correlation three ways, plus the z-score identity
r_pandas = df.study_hours.corr(df.marks)
r_scipy, p_value = stats.pearsonr(df.study_hours, df.marks)
zx = (df.study_hours - df.study_hours.mean()) / df.study_hours.std(ddof=1)
zy = (df.marks - df.marks.mean()) / df.marks.std(ddof=1)
r_z = (zx * zy).sum() / (n - 1)

print(f"r (pandas)        : {r_pandas:.6f}")
print(f"r (scipy)         : {r_scipy:.6f}   p-value = {p_value:.3e}")
print(f"r (mean of z*z)   : {r_z:.6f}")
print(f"r^2               : {r_scipy**2:.4f}  -> {r_scipy**2*100:.1f}% of the variance in marks")

In [ ]:
# A gallery of correlation strengths
def correlated_pair(r, m=300, seed=0):
    '''Generate m points with an (approximately) specified Pearson r.'''
    g = np.random.default_rng(seed)
    x = g.normal(size=m)
    e = g.normal(size=m)
    y = r * x + np.sqrt(max(0.0, 1 - r**2)) * e
    return x, y

targets = [0.99, 0.8, 0.5, 0.2, 0.0, -0.5, -0.9, -0.99]
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for ax, r_target in zip(axes.ravel(), targets):
    x, y = correlated_pair(r_target, seed=int(abs(r_target) * 100) + 1)
    ax.scatter(x, y, s=10, alpha=0.6, color="steelblue")
    ax.set_title(f"target r = {r_target:+.2f}\nactual r = {np.corrcoef(x, y)[0,1]:+.2f}"
                 f"\nr^2 = {np.corrcoef(x, y)[0,1]**2:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

---
## 5.3 What correlation misses

Pearson $r$ measures **linear** association only. Three failure modes, all common:

1. **Non-linearity** — a perfect curved relationship can give $r \approx 0$
2. **Outliers** — one point can create or destroy a correlation
3. **Different shapes, identical $r$** — Anscombe's quartet

> **Rule you should never break: plot your data before you trust a correlation.**

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))

x = np.linspace(-3, 3, 200)
cases = [
    ("Perfect parabola",  x, x**2 + rng.normal(0, 0.2, 200)),
    ("Sine wave",         x, np.sin(x * 2) + rng.normal(0, 0.1, 200)),
    ("Two clusters",      np.r_[rng.normal(-2, .4, 100), rng.normal(2, .4, 100)],
                          np.r_[rng.normal(2, .4, 100),  rng.normal(2, .4, 100)]),
    ("Heteroscedastic",   x, x * rng.normal(0, 1, 200)),
]
for ax, (name, xx, yy) in zip(axes, cases):
    r = np.corrcoef(xx, yy)[0, 1]
    ax.scatter(xx, yy, s=12, alpha=0.6, color="steelblue")
    ax.set_title(f"{name}\nr = {r:+.3f}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print("Every panel shows an obvious structure, and r is near zero in each.")
print("r = 0 means 'no LINEAR relationship', not 'no relationship'.")

In [ ]:
# The outlier problem, in both directions
base_x = rng.normal(0, 1, 60)
base_y = rng.normal(0, 1, 60)              # genuinely uncorrelated

x_out = np.append(base_x, 12)
y_out = np.append(base_y, 12)              # one extreme point

lin_x = np.arange(60.0)
lin_y = lin_x + rng.normal(0, 3, 60)       # genuinely correlated
lin_x2 = np.append(lin_x, 30); lin_y2 = np.append(lin_y, 400)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].scatter(base_x, base_y, s=20, color="steelblue", label=f"without outlier: r={np.corrcoef(base_x, base_y)[0,1]:+.2f}")
ax[0].scatter([12], [12], s=70, color="crimson", label=f"with outlier: r={np.corrcoef(x_out, y_out)[0,1]:+.2f}")
ax[0].legend(fontsize=8); ax[0].set_title("One point CREATES a correlation")

ax[1].scatter(lin_x, lin_y, s=20, color="steelblue", label=f"without outlier: r={np.corrcoef(lin_x, lin_y)[0,1]:+.2f}")
ax[1].scatter([30], [400], s=70, color="crimson", label=f"with outlier: r={np.corrcoef(lin_x2, lin_y2)[0,1]:+.2f}")
ax[1].legend(fontsize=8); ax[1].set_title("One point DESTROYS a correlation")
plt.tight_layout(); plt.show()

### Anscombe's quartet

Frank Anscombe (1973) constructed four datasets with **identical** means, variances,
correlations and regression lines — and utterly different shapes. It remains the best
one-slide argument for plotting your data.

In [ ]:
anscombe = pd.DataFrame({
    "x1": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "y1": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    "x2": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "y2": [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    "x3": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "y3": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    "x4": [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y4": [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})

summary = []
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for i, ax in enumerate(axes, start=1):
    xx, yy = anscombe[f"x{i}"], anscombe[f"y{i}"]
    slope, intercept, r, p, se = stats.linregress(xx, yy)
    ax.scatter(xx, yy, s=45, color="steelblue", zorder=3)
    xs = np.linspace(3, 20, 10)
    ax.plot(xs, intercept + slope * xs, color="crimson", lw=1.6)
    ax.set_xlim(2, 20); ax.set_ylim(2, 14)
    ax.set_title(f"Dataset {i}\nr = {r:.3f}", fontsize=10)
    summary.append({"dataset": i, "mean x": xx.mean(), "mean y": round(yy.mean(), 2),
                    "sd y": round(yy.std(ddof=1), 2), "r": round(r, 3),
                    "slope": round(slope, 2), "intercept": round(intercept, 2)})
plt.tight_layout(); plt.show()

print(pd.DataFrame(summary).to_string(index=False))
print("\nIdentical summary statistics. Dataset 1 is a genuine linear relationship;")
print("2 is a curve; 3 is a line plus one outlier; 4 is one influential point doing all the work.")

---
## 5.4 Rank correlations: Spearman and Kendall

When the relationship is **monotonic but not linear**, or the data has outliers or is
ordinal (survey scales, rankings), use a rank-based measure.

**Spearman's $\rho$** — Pearson correlation of the *ranks*. Captures any monotonic
relationship; robust to outliers.

**Kendall's $\tau$** — based on counting **concordant** and **discordant** pairs:

$$\tau = \frac{(\#\text{concordant}) - (\#\text{discordant})}{\binom{n}{2}}$$

Kendall is more interpretable (a probability difference), more robust for small $n$, but
slower to compute.

**Choosing:**

| Situation | Use |
|---|---|
| Linear, roughly Normal, no big outliers | Pearson |
| Monotonic but curved | Spearman |
| Ordinal data / ranks / small n / many ties | Kendall |
| Non-monotonic relationship | None of them — plot it, or use mutual information |

In [ ]:
# A perfectly monotonic but strongly non-linear relationship
x = np.linspace(1, 10, 60)
y = np.exp(x / 1.6)

print("Perfect monotonic (exponential) relationship:")
print(f"  Pearson  r    = {stats.pearsonr(x, y)[0]:.4f}   <- understates it")
print(f"  Spearman rho  = {stats.spearmanr(x, y)[0]:.4f}   <- correctly reports 1.0")
print(f"  Kendall  tau  = {stats.kendalltau(x, y)[0]:.4f}   <- also 1.0")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, y, color="steelblue"); ax[0].set_title("Raw values: curved")
ax[1].scatter(stats.rankdata(x), stats.rankdata(y), color="seagreen")
ax[1].set_title("Ranks: perfectly straight")
ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].set_xlabel("rank of x"); ax[1].set_ylabel("rank of y")
plt.tight_layout(); plt.show()

In [ ]:
# Robustness to a single outlier
clean_x = rng.normal(0, 1, 80)
clean_y = 0.8 * clean_x + rng.normal(0, 0.6, 80)
dirty_x = np.append(clean_x, 9.0)
dirty_y = np.append(clean_y, -9.0)          # one wild point in the wrong direction

rows = []
for name, (xx, yy) in [("clean", (clean_x, clean_y)), ("with 1 outlier", (dirty_x, dirty_y))]:
    rows.append({
        "data": name,
        "Pearson":  round(stats.pearsonr(xx, yy)[0], 3),
        "Spearman": round(stats.spearmanr(xx, yy)[0], 3),
        "Kendall":  round(stats.kendalltau(xx, yy)[0], 3),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nPearson collapses; the rank measures barely move. That is robustness.")

---
## 5.5 Correlation matrices and heatmaps

With $k$ numeric columns you get a $k \times k$ symmetric matrix. A heatmap makes it
readable. Use a **diverging** colour map centred on zero so that positive and negative
correlations are visually distinct, and always fix `vmin=-1, vmax=1` so colours mean the
same thing across plots.

In [ ]:
from sklearn.datasets import load_diabetes

dia = load_diabetes(as_frame=True)
data = dia.frame.rename(columns={"target": "disease_progression"})
print(f"{data.shape[0]} patients, {data.shape[1]} columns")
print(dia.DESCR.split("**Data Set Characteristics:**")[0][:300])
data.head()

In [ ]:
corr = data.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)     # hide the mirrored half
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation matrix (diabetes dataset)")
plt.tight_layout(); plt.show()

In [ ]:
# Which features correlate most with the target? A standard first-pass feature screen.
target_corr = (corr["disease_progression"]
               .drop("disease_progression")
               .sort_values(key=abs, ascending=False))

print("Correlation with disease progression, strongest first:")
for feature, r in target_corr.items():
    bar = "#" * int(abs(r) * 40)
    print(f"  {feature:>6}  {r:+.3f}  {bar}")

print("\nCaution: this screen is univariate. A feature with r = 0.05 on its own can still")
print("be valuable in combination with others, and two features with r = 0.6 each may be")
print("carrying the same information twice.")

In [ ]:
# Pairwise scatter plots for the strongest few features
top = list(target_corr.index[:3]) + ["disease_progression"]
sns.pairplot(data[top], plot_kws={"s": 14, "alpha": 0.6}, diag_kind="kde", height=1.9)
plt.suptitle("Pair plot of the strongest features", y=1.01)
plt.show()

---
## 5.6 Is the correlation statistically significant?

A correlation computed from 10 points is not the same evidence as one from 10,000. The test
statistic is

$$t = r\sqrt{\frac{n-2}{1-r^2}} \sim t_{n-2} \quad \text{under } H_0: \rho = 0$$

`scipy.stats.pearsonr` returns the p-value for you. Two warnings:

- **Small $n$:** even $r = 0.6$ can be non-significant with $n = 8$
- **Large $n$:** $r = 0.03$ becomes "significant" with $n = 100{,}000$ but is useless

Statistical significance answers *"is it distinguishable from zero?"*, not *"does it
matter?"* Notebooks 6–8 develop this distinction properly.

In [ ]:
print(f"{'n':>8} {'r':>7} {'t':>8} {'p-value':>12}  significant at 5%?")
for m, r_true in [(8, 0.6), (30, 0.35), (100, 0.20), (1_000, 0.06), (100_000, 0.01)]:
    xx, yy = correlated_pair(r_true, m=m, seed=m)
    r_obs, p = stats.pearsonr(xx, yy)
    t = r_obs * np.sqrt((m - 2) / (1 - r_obs**2))
    print(f"{m:>8} {r_obs:>+7.3f} {t:>8.2f} {p:>12.2e}  {'YES' if p < 0.05 else 'no'}")
print("\nBottom row: significant, and completely unimportant (r^2 = 0.0001).")

In [ ]:
# Confidence interval for r via Fisher's z-transformation
def r_confidence_interval(r, m, conf=0.95):
    '''Fisher z-transform CI for a Pearson correlation.'''
    z = np.arctanh(r)
    se = 1 / np.sqrt(m - 3)
    crit = stats.norm.ppf(1 - (1 - conf) / 2)
    return np.tanh([z - crit * se, z + crit * se])

for m in (10, 30, 100, 1000):
    lo, hi = r_confidence_interval(0.5, m)
    print(f"r = 0.50 with n = {m:>4}: 95% CI = [{lo:+.3f}, {hi:+.3f}]  (width {hi-lo:.3f})")
print("\nWith n = 10 the interval spans almost everything -- the estimate is nearly useless.")

---
## 5.7 Correlation does not imply causation

If $X$ and $Y$ are correlated, there are at least five explanations:

1. $X$ causes $Y$
2. $Y$ causes $X$ (**reverse causation**)
3. A third variable $Z$ causes both (**confounding**)
4. Selection effects in how the data was gathered (**collider bias**)
5. Pure coincidence (**spurious correlation**), especially with many variables tested

Only a randomised experiment, or careful causal modelling, distinguishes them.

In [ ]:
# CONFOUNDING: ice cream sales and drowning deaths are correlated -- because of temperature.
m = 365
temperature = 18 + 12 * np.sin(np.linspace(0, 2*np.pi, m)) + rng.normal(0, 2, m)
ice_cream   = 50 + 8 * temperature + rng.normal(0, 25, m)
drownings   = 1 + 0.35 * temperature + rng.normal(0, 1.4, m)

conf = pd.DataFrame({"temperature": temperature, "ice_cream": ice_cream, "drownings": drownings})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(conf.ice_cream, conf.drownings, s=12, alpha=0.6, color="crimson")
ax[0].set_title(f"Ice cream vs drownings\nr = {conf.ice_cream.corr(conf.drownings):.3f}")
ax[0].set_xlabel("ice cream sales"); ax[0].set_ylabel("drownings")

ax[1].scatter(conf.temperature, conf.ice_cream, s=12, alpha=0.6, color="steelblue")
ax[1].set_title(f"Temperature -> ice cream\nr = {conf.temperature.corr(conf.ice_cream):.3f}")
ax[1].set_xlabel("temperature")

ax[2].scatter(conf.temperature, conf.drownings, s=12, alpha=0.6, color="steelblue")
ax[2].set_title(f"Temperature -> drownings\nr = {conf.temperature.corr(conf.drownings):.3f}")
ax[2].set_xlabel("temperature")
plt.tight_layout(); plt.show()

print("Banning ice cream would not save a single swimmer.")
print("Temperature is the common cause; the left-hand correlation is entirely derivative.")

### Partial correlation: controlling for a confounder

**Partial correlation** $r_{XY \cdot Z}$ measures the correlation between $X$ and $Y$
*after removing the linear influence of $Z$ from both*. The recipe:

1. Regress $X$ on $Z$, keep the residuals
2. Regress $Y$ on $Z$, keep the residuals
3. Correlate the two residual series

If the original correlation was entirely due to $Z$, the partial correlation collapses to
about zero.

In [ ]:
def partial_corr(x, y, z):
    '''Correlation between x and y after linearly removing z from both.'''
    res_x = x - np.polyval(np.polyfit(z, x, 1), z)
    res_y = y - np.polyval(np.polyfit(z, y, 1), z)
    return stats.pearsonr(res_x, res_y)

r_raw, p_raw = stats.pearsonr(conf.ice_cream, conf.drownings)
r_par, p_par = partial_corr(conf.ice_cream.values, conf.drownings.values, conf.temperature.values)

print(f"Raw correlation      r(ice cream, drownings)              = {r_raw:+.4f}  (p={p_raw:.1e})")
print(f"Partial correlation  r(ice cream, drownings | temperature) = {r_par:+.4f}  (p={p_par:.3f})")
print("\nControlling for temperature makes the association vanish -- the signature of a confounder.")

### Simpson's paradox

The most dramatic failure: a relationship that holds in **every subgroup** can **reverse**
when the groups are pooled. This is not a curiosity — it appears in medical trials,
university admissions, and A/B tests all the time.

In [ ]:
# Two departments. In BOTH, treatment B has a higher success rate.
# But B is mostly applied in the hard department, so pooling flips the answer.
records = []
for dept, base, n_a, n_b in [("Easy", 0.90, 500, 50), ("Hard", 0.30, 50, 500)]:
    for treat, lift in [("A", 0.00), ("B", 0.05)]:
        k = n_a if treat == "A" else n_b
        succ = rng.random(k) < (base + lift)
        records.append(pd.DataFrame({"dept": dept, "treatment": treat, "success": succ.astype(int)}))
sp = pd.concat(records, ignore_index=True)

print("WITHIN each department (B wins both times):")
print(sp.groupby(["dept", "treatment"])["success"].agg(["count", "mean"]).round(3))
print("\nPOOLED across departments (A appears to win):")
print(sp.groupby("treatment")["success"].agg(["count", "mean"]).round(3))
print("\nB looks worse overall only because it was mostly used on hard cases.")
print("The fix is to compare within strata, or to randomise assignment.")

In [ ]:
# SPURIOUS CORRELATION: test enough random variable pairs and you will find "significant" ones.
n_vars, m = 100, 40
noise = rng.normal(size=(m, n_vars))
cm = np.corrcoef(noise, rowvar=False)
off_diag = cm[np.triu_indices(n_vars, k=1)]

n_pairs = len(off_diag)
significant = 0
for i in range(n_vars):
    for j in range(i + 1, n_vars):
        if stats.pearsonr(noise[:, i], noise[:, j])[1] < 0.05:
            significant += 1

print(f"{n_vars} pure-noise variables -> {n_pairs:,} pairs tested")
print(f"Strongest |r| found by chance : {np.abs(off_diag).max():.3f}")
print(f"'Significant' at p < 0.05     : {significant:,} pairs ({significant/n_pairs:.1%})")
print(f"Expected by chance alone      : {0.05*n_pairs:.0f} pairs (5%)")
print("\nThis is the MULTIPLE COMPARISONS problem. Hunting through a correlation matrix")
print("for the biggest number is a reliable way to find nothing real (see Notebook 6).")

---
## 5.8 Multicollinearity: when features correlate with *each other*

For prediction, correlated features are mostly a nuisance. For **interpretation**, they are
a disaster: when two predictors carry the same information, the regression cannot tell
which one deserves the credit, so the coefficients become wildly unstable — large standard
errors, and signs that flip when you add one more row of data.

The standard diagnostic is the **Variance Inflation Factor**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ comes from regressing feature $j$ on all the *other* features.

| VIF | Reading |
|---|---|
| 1 | no collinearity |
| 1 – 5 | moderate, usually fine |
| 5 – 10 | high, investigate |
| > 10 | severe — drop, combine, or regularise |

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

# x2 is almost a copy of x1
m = 300
x1 = rng.normal(0, 1, m)
x2 = x1 + rng.normal(0, 0.08, m)          # r(x1, x2) approx 0.997
x3 = rng.normal(0, 1, m)
y  = 3 * x1 + 2 * x3 + rng.normal(0, 1, m)   # x2 has NO true effect

Xc = np.column_stack([x1, x2, x3])
print(f"r(x1, x2) = {np.corrcoef(x1, x2)[0,1]:.4f}\n")

def vif(X):
    out = []
    for j in range(X.shape[1]):
        others = np.delete(X, j, axis=1)
        r2 = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
        out.append(1 / (1 - r2))
    return out

print("VIF:", [f"{v:.1f}" for v in vif(Xc)])
print("\nTrue coefficients: x1 = 3.0, x2 = 0.0, x3 = 2.0")
print("Fitted on 8 bootstrap resamples -- watch x1 and x2 swing wildly:")
for i in range(8):
    idx = rng.integers(0, m, m)
    c = LinearRegression().fit(Xc[idx], y[idx]).coef_
    print(f"   x1={c[0]:+7.2f}   x2={c[1]:+7.2f}   x3={c[2]:+7.2f}   (x1+x2={c[0]+c[1]:+6.2f})")
print("\nThe SUM x1+x2 is stable at about 3 -- the model knows the total effect,")
print("it just cannot split it. Ridge regularisation or dropping one feature fixes this.")

In [ ]:
# Ridge shrinks the unstable pair toward a stable compromise
print("Ridge (alpha=10) on the same 8 resamples:")
for i in range(8):
    idx = rng.integers(0, m, m)
    c = Ridge(alpha=10).fit(Xc[idx], y[idx]).coef_
    print(f"   x1={c[0]:+7.2f}   x2={c[1]:+7.2f}   x3={c[2]:+7.2f}")
print("\nFar less variance across resamples (at the cost of a little bias).")

In [ ]:
# A practical workflow: flag highly correlated feature pairs before modelling
def high_corr_pairs(frame, threshold=0.7):
    c = frame.corr(numeric_only=True).abs()
    upper = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1))
    pairs = (upper.stack()
                  .sort_values(ascending=False)
                  .rename("abs_r")
                  .reset_index()
                  .rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
    return pairs[pairs.abs_r >= threshold]

flagged = high_corr_pairs(data.drop(columns="disease_progression"), threshold=0.5)
print("Feature pairs with |r| >= 0.5 in the diabetes data:")
print(flagged.round(3).to_string(index=False) if len(flagged) else "  none")

---
## Exercises

**Exercise 1.** For the following dataset, compute covariance, Pearson $r$, Spearman $\rho$
and $r^2$. Then plot it and explain which statistic best describes the relationship.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
ad_spend = np.array([1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 70, 95])
revenue  = np.array([12, 22, 31, 46, 66, 88, 112, 134, 152, 168, 178, 185])

print(f"Covariance   : {np.cov(ad_spend, revenue, ddof=1)[0,1]:.2f}")
print(f"Pearson r    : {stats.pearsonr(ad_spend, revenue)[0]:.4f}  (r^2 = {stats.pearsonr(ad_spend, revenue)[0]**2:.4f})")
print(f"Spearman rho : {stats.spearmanr(ad_spend, revenue)[0]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ad_spend, revenue, "o-", color="steelblue")
ax[0].set_xlabel("ad spend"); ax[0].set_ylabel("revenue"); ax[0].set_title("Raw: clearly curved")
ax[1].plot(np.log(ad_spend), revenue, "o-", color="seagreen")
ax[1].set_xlabel("log(ad spend)"); ax[1].set_title(
    f"After log transform: r = {stats.pearsonr(np.log(ad_spend), revenue)[0]:.4f}")
plt.tight_layout(); plt.show()

print("\nSpearman = 1.0 because the relationship is perfectly monotonic.")
print("Pearson understates it because of diminishing returns (saturation).")
print("Log-transforming the spend restores linearity -- the right fix in practice.")

**Exercise 2.** Load the iris dataset. Produce the correlation heatmap, identify the most
and least correlated feature pairs, and then compute the correlation of petal length with
petal width **separately within each species**. Compare with the pooled correlation. What
do you notice?

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True).frame
iris["species"] = load_iris().target_names[iris["target"]]
feat = [c for c in iris.columns if "(cm)" in c]

c = iris[feat].corr()
sns.heatmap(c, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1, center=0, square=True)
plt.title("Iris feature correlations"); plt.tight_layout(); plt.show()

up = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1)).stack()
print(f"Strongest pair : {up.idxmax()} r = {up.max():.3f}")
print(f"Weakest pair   : {up.abs().idxmin()} r = {up[up.abs().idxmin()]:.3f}\n")

print("r(petal length, petal width):")
print(f"  pooled across species : {iris['petal length (cm)'].corr(iris['petal width (cm)']):.3f}")
for sp, g in iris.groupby("species"):
    print(f"  within {sp:<12}   : {g['petal length (cm)'].corr(g['petal width (cm)']):.3f}")
print("\nThe pooled correlation (0.96) is far stronger than any within-species one.")
print("It is driven mostly by BETWEEN-species differences in size -- a mild Simpson effect.")
print("Always ask whether a correlation lives within groups or between them.")

**Exercise 3.** Simulate a confounded system: `experience -> salary` and
`experience -> years_at_company`, with no direct link between salary and tenure.
(a) Show that salary and tenure appear correlated.
(b) Show the partial correlation controlling for experience is near zero.
(c) Explain what would go wrong if a manager used tenure to set salaries.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
m = 400
experience = rng.uniform(0, 30, m)
salary     = 300_000 + 45_000 * experience + rng.normal(0, 120_000, m)
tenure     = np.clip(0.55 * experience + rng.normal(0, 1.8, m), 0, None)

r_raw = stats.pearsonr(salary, tenure)
r_par = partial_corr(salary, tenure, experience)

print(f"(a) r(salary, tenure)                = {r_raw[0]:+.4f}  (p = {r_raw[1]:.2e})")
print(f"(b) r(salary, tenure | experience)   = {r_par[0]:+.4f}  (p = {r_par[1]:.3f})")
print()
print("(c) Tenure has no causal effect on salary here -- both are driven by experience.")
print("    Rewarding tenure would overpay long-serving but inexperienced staff and")
print("    underpay experienced new hires. Correlation identified the wrong lever.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(tenure, salary, s=12, alpha=0.6, color="crimson")
ax[0].set_title(f"Looks causal: r = {r_raw[0]:.3f}")
ax[0].set_xlabel("years at company"); ax[0].set_ylabel("salary")
rx = tenure - np.polyval(np.polyfit(experience, tenure, 1), experience)
ry = salary - np.polyval(np.polyfit(experience, salary, 1), experience)
ax[1].scatter(rx, ry, s=12, alpha=0.6, color="steelblue")
ax[1].set_title(f"Residuals after removing experience: r = {r_par[0]:.3f}")
ax[1].set_xlabel("tenure residual"); ax[1].set_ylabel("salary residual")
plt.tight_layout(); plt.show()

**Exercise 4 (challenge).** You are handed 50 candidate features and one target, and asked
to "pick the features with the highest correlation to the target". Explain, with a
simulation, why choosing features on the *full* dataset before splitting inflates your
reported accuracy — even when none of the features carries real signal.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression

m, n_feat = 120, 50
Xr = rng.normal(size=(m, n_feat))
yr = rng.normal(size=m)                     # target is PURE NOISE -- no real signal at all

# WRONG: screen features using all the data, then evaluate
r_all = np.array([abs(np.corrcoef(Xr[:, j], yr)[0, 1]) for j in range(n_feat)])
best = np.argsort(r_all)[-5:]
wrong = cross_val_score(LinearRegression(), Xr[:, best], yr, cv=5, scoring="r2").mean()

# RIGHT: split first, screen on the training half only, evaluate on the held-out half
X_tr, X_te, y_tr, y_te = train_test_split(Xr, yr, test_size=0.4, random_state=0)
r_tr = np.array([abs(np.corrcoef(X_tr[:, j], y_tr)[0, 1]) for j in range(n_feat)])
best_tr = np.argsort(r_tr)[-5:]
model = LinearRegression().fit(X_tr[:, best_tr], y_tr)
right = model.score(X_te[:, best_tr], y_te)

print(f"Top 5 |r| found on the full data : {np.round(r_all[best], 3)}")
print(f"\nR^2 selecting on ALL data (leaky) : {wrong:+.3f}")
print(f"R^2 selecting on TRAIN only      : {right:+.3f}")
print("\nThere is no signal in this data at all. The leaky pipeline still reports a")
print("respectable R^2, because with 50 features and n=120 some will correlate with the")
print("target by chance -- and the same lucky noise is then 'validated' on the same rows.")
print("Feature selection is part of the model. It must happen INSIDE the split.")

---
## Summary

| Concept | Formula / key point |
|---|---|
| Covariance | $\operatorname{Cov}(X,Y) = E[XY]-E[X]E[Y]$; sign only, units meaningless |
| Pearson $r$ | $\operatorname{Cov}(X,Y)/(\sigma_X\sigma_Y)$, in $[-1,1]$; **linear** only |
| $r^2$ | Fraction of variance explained — always report it |
| Spearman $\rho$ | Pearson on ranks; monotonic, robust |
| Kendall $\tau$ | Concordant minus discordant pairs; best for ordinal / small $n$ |
| Significance | $t = r\sqrt{(n-2)/(1-r^2)}$; large $n$ makes trivial $r$ "significant" |
| Fisher $z$ CI | $z = \operatorname{arctanh}(r)$, $\operatorname{SE} = 1/\sqrt{n-3}$ |
| Partial correlation | Correlate the residuals after removing $Z$ |
| Simpson's paradox | Subgroup and pooled relationships can point opposite ways |
| VIF | $1/(1-R_j^2)$; over 10 means severe multicollinearity |
| The golden rule | **Always plot the data** |

**Next up:** [Notebook 6 — Hypothesis Testing](6.%20Hypothesis%20Testing.ipynb), the framework
for deciding whether a pattern is real or just noise.